# Experiment A: Consistency Metric Across All Domains and Properties

**Purpose:** Add the accuracy-consistency gap metric to every domain in EqRelBench.

The EqRelBench paper tests 5 semantic domains (kinship, set theory, object identity, synonymy, spatial) plus an abstract random-name baseline. It currently only reports per-question accuracy. This notebook computes **both** accuracy and logical consistency for each property × domain combination, producing the full table the paper needs.

## Domains tested
| Domain | Relation template | Example |
|---|---|---|
| Abstract | `equals` | "KFQZ equals ABLM" |
| Kinship | `is the same age as` | "KFQZ is the same age as ABLM" |
| Set Theory | `has the same cardinality as` | "Set KFQZ has the same cardinality as Set ABLM" |
| Object Identity | `is identical to` | "Object KFQZ is identical to Object ABLM" |
| Synonymy | `is synonymous with` | "Term KFQZ is synonymous with Term ABLM" |
| Spatial | `is at the same location as` | "Location KFQZ is at the same location as Location ABLM" |

## Output
- Table: Accuracy + Consistency + Gap for each property × domain (fills EqRelBench Table 1)
- Figure: Heatmap of accuracy-consistency gap across all conditions
- JSON: `experiment_A_results.json`

In [ ]:
!git clone https://github.com/Atharvy700/SP-25.git 2>/dev/null || echo 'Already cloned'
%cd SP-25
!pip install transformers accelerate -q

In [ ]:
import json, random, gc, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict

random.seed(42)
np.random.seed(42)

print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Entity name pool (same method as repo) ───────────────────────────────────
def gen_names(n=50000, lo=4, hi=7, seed=42):
    rng = random.Random(seed)
    pool = set()
    while len(pool) < n:
        k = rng.randint(lo, hi)
        pool.add(''.join(rng.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=k)))
    return list(pool)

NAMES = gen_names()
print(f'Name pool: {len(NAMES)} entities')
print(f'Examples: {NAMES[:6]}')

In [ ]:
# ── Domain templates ─────────────────────────────────────────────────────────
# Each domain specifies how to express premises and questions.
# The relational structure is identical across all domains;
# only the surface lexical form changes. This lets us test whether
# accuracy and consistency failures are structural or domain-specific.

DOMAINS = {
    'abstract': {
        'premise':    '{A} equals {B}.',
        'question':   'Does {A} equal {B}?',
        'refl_q':     'Does {A} equal {A}?',
        'prefix_A':   '',
        'prefix_B':   '',
    },
    'kinship': {
        'premise':    '{A} is the same age as {B}.',
        'question':   'Is {A} the same age as {B}?',
        'refl_q':     'Is {A} the same age as {A}?',
        'prefix_A':   '',
        'prefix_B':   '',
    },
    'set_theory': {
        'premise':    'Set {A} has the same cardinality as set {B}.',
        'question':   'Does set {A} have the same cardinality as set {B}?',
        'refl_q':     'Does set {A} have the same cardinality as set {A}?',
        'prefix_A':   'Set ',
        'prefix_B':   'set ',
    },
    'object_identity': {
        'premise':    'Object {A} is identical to object {B}.',
        'question':   'Is object {A} identical to object {B}?',
        'refl_q':     'Is object {A} identical to object {A}?',
        'prefix_A':   'Object ',
        'prefix_B':   'object ',
    },
    'synonymy': {
        'premise':    'The term {A} is synonymous with the term {B}.',
        'question':   'Is the term {A} synonymous with the term {B}?',
        'refl_q':     'Is the term {A} synonymous with the term {A}?',
        'prefix_A':   'the term ',
        'prefix_B':   'the term ',
    },
    'spatial': {
        'premise':    'Location {A} is at the same coordinates as location {B}.',
        'question':   'Is location {A} at the same coordinates as location {B}?',
        'refl_q':     'Is location {A} at the same coordinates as location {A}?',
        'prefix_A':   'Location ',
        'prefix_B':   'location ',
    },
}

print('Domains configured:', list(DOMAINS.keys()))

In [ ]:
# ── Prompt construction ───────────────────────────────────────────────────────
BASE_INSTRUCTION = (
    'You are a logical reasoning assistant.\n'
    'Answer with exactly one word.\n\n'
)
ANSWER_SUFFIX = '\n\nAnswer only yes or no:'


def build_sym_prompts(A, B, domain):
    """Returns (prompt_fwd, prompt_rev) for a symmetry consistency pair."""
    d = DOMAINS[domain]
    premise_AB = 'Suppose ' + d['premise'].format(A=A, B=B)
    premise_BA = 'Suppose ' + d['premise'].format(A=B, B=A)
    q_BA = d['question'].format(A=B, B=A)   # forward: premise AB, ask BA
    q_AB = d['question'].format(A=A, B=B)   # reverse: premise BA, ask AB

    fwd = BASE_INSTRUCTION + premise_AB + ' ' + q_BA + ANSWER_SUFFIX
    rev = BASE_INSTRUCTION + premise_BA + ' ' + q_AB + ANSWER_SUFFIX
    return fwd, rev


def build_trans_prompts(A, B, C, domain):
    """Returns (prompt_direct, prompt_reorder, prompt_reverse) for a transitivity triple."""
    d = DOMAINS[domain]
    p_AB = 'Suppose ' + d['premise'].format(A=A, B=B)
    p_BC = 'Suppose ' + d['premise'].format(A=B, B=C)
    q_AC = d['question'].format(A=A, B=C)   # direct: A to C
    q_CA = d['question'].format(A=C, B=A)   # reverse: C to A (requires sym+trans)

    direct  = BASE_INSTRUCTION + p_AB + ' ' + p_BC + ' ' + q_AC + ANSWER_SUFFIX
    reorder = BASE_INSTRUCTION + p_BC + ' ' + p_AB + ' ' + q_AC + ANSWER_SUFFIX  # premises swapped
    reverse = BASE_INSTRUCTION + p_AB + ' ' + p_BC + ' ' + q_CA + ANSWER_SUFFIX  # conclusion reversed
    return direct, reorder, reverse


def build_refl_prompt(X, domain):
    """Returns prompt for reflexivity: Does X R X?"""
    d = DOMAINS[domain]
    q = d['refl_q'].format(A=X)
    return BASE_INSTRUCTION + q + ANSWER_SUFFIX


# Sanity check
A, B, C = NAMES[0], NAMES[1], NAMES[2]
fwd, rev = build_sym_prompts(A, B, 'kinship')
print('Symmetry prompt (fwd):')
print(fwd)
print()
d, ro, rv = build_trans_prompts(A, B, C, 'set_theory')
print('Transitivity prompt (direct):')
print(d)

In [ ]:
# ── Load Phi-3-mini ───────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'
MAX_LENGTH  = 384

print('Loading tokenizer...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print('Loading model (FP16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model.eval()

YES_ID = tok('yes', add_special_tokens=False).input_ids[0]
NO_ID  = tok('no',  add_special_tokens=False).input_ids[0]
print(f'yes_id={YES_ID}  no_id={NO_ID}')
print('Model ready.')

In [ ]:
# ── Batched inference ─────────────────────────────────────────────────────────
@torch.inference_mode()
def predict_batch(prompts, batch_size=8):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(model.device)
        logits = model(**enc).logits[:, -1, :]
        for lg in logits:
            results.append('yes' if lg[YES_ID] > lg[NO_ID] else 'no')
    return results

print('Inference helper ready.')

In [ ]:
# ── Per-domain evaluation functions ──────────────────────────────────────────
N_SYM   = 200   # pairs per domain
N_TRANS = 150   # triples per domain
N_REFL  = 200   # entities per domain

rng = random.Random(42)
pool = NAMES[:8000]


def eval_symmetry(domain):
    pairs = []
    used = set()
    while len(pairs) < N_SYM:
        A, B = rng.sample(pool, 2)
        if (A, B) not in used:
            used.add((A, B))
            pairs.append((A, B))

    fwd_prompts = [build_sym_prompts(A, B, domain)[0] for A, B in pairs]
    rev_prompts = [build_sym_prompts(A, B, domain)[1] for A, B in pairs]

    preds_fwd = predict_batch(fwd_prompts)
    preds_rev = predict_batch(rev_prompts)

    acc_fwd  = np.mean([p == 'yes' for p in preds_fwd])
    acc_rev  = np.mean([p == 'yes' for p in preds_rev])
    avg_acc  = (acc_fwd + acc_rev) / 2
    cons     = np.mean([pf == pr for pf, pr in zip(preds_fwd, preds_rev)])
    gap      = avg_acc - cons

    # Failure breakdown
    fail_types = defaultdict(int)
    for pf, pr in zip(preds_fwd, preds_rev):
        if pf == 'yes' and pr == 'yes':   fail_types['both_yes']  += 1
        elif pf == 'yes' and pr == 'no':  fail_types['fwd_only']  += 1
        elif pf == 'no'  and pr == 'yes': fail_types['rev_only']  += 1
        else:                              fail_types['both_no']   += 1

    return {'avg_acc': avg_acc, 'acc_fwd': acc_fwd, 'acc_rev': acc_rev,
            'consistency': cons, 'gap': gap, 'fail_types': dict(fail_types), 'n': N_SYM}


def eval_transitivity(domain):
    triples = []
    used = set()
    while len(triples) < N_TRANS:
        A, B, C = rng.sample(pool, 3)
        if (A, B, C) not in used:
            used.add((A, B, C))
            triples.append((A, B, C))

    d_prompts  = [build_trans_prompts(A, B, C, domain)[0] for A, B, C in triples]
    ro_prompts = [build_trans_prompts(A, B, C, domain)[1] for A, B, C in triples]
    rv_prompts = [build_trans_prompts(A, B, C, domain)[2] for A, B, C in triples]

    preds_d  = predict_batch(d_prompts)
    preds_ro = predict_batch(ro_prompts)
    preds_rv = predict_batch(rv_prompts)

    acc_d   = np.mean([p == 'yes' for p in preds_d])
    acc_ro  = np.mean([p == 'yes' for p in preds_ro])
    acc_rv  = np.mean([p == 'yes' for p in preds_rv])
    avg_acc = (acc_d + acc_ro + acc_rv) / 3

    cons_d_ro = np.mean([pd == pro for pd, pro in zip(preds_d, preds_ro)])
    cons_d_rv = np.mean([pd == prv for pd, prv in zip(preds_d, preds_rv)])
    all_cons  = np.mean([pd == pro == prv
                         for pd, pro, prv in zip(preds_d, preds_ro, preds_rv)])
    gap = avg_acc - all_cons

    viol_ro = np.mean([pd == 'yes' and pro == 'no'
                       for pd, pro in zip(preds_d, preds_ro)])
    viol_rv = np.mean([pd == 'yes' and prv == 'no'
                       for pd, prv in zip(preds_d, preds_rv)])

    return {'avg_acc': avg_acc, 'acc_direct': acc_d, 'acc_reorder': acc_ro,
            'acc_reverse': acc_rv, 'consistency': all_cons,
            'cons_d_ro': cons_d_ro, 'cons_d_rv': cons_d_rv,
            'gap': gap, 'viol_reorder': viol_ro, 'viol_reverse': viol_rv, 'n': N_TRANS}


def eval_reflexivity(domain):
    entities = rng.sample(pool, N_REFL)
    prompts  = [build_refl_prompt(X, domain) for X in entities]
    preds    = predict_batch(prompts)
    acc      = np.mean([p == 'yes' for p in preds])  # all should be yes
    # For reflexivity, consistency = accuracy (each question is independent)
    return {'avg_acc': acc, 'consistency': acc, 'gap': 0.0,
            'n_yes': sum(p == 'yes' for p in preds), 'n': N_REFL}


print('Evaluation functions ready.')

In [ ]:
# ── RUN ALL DOMAINS × ALL PROPERTIES ─────────────────────────────────────────
all_results = {}

for domain in DOMAINS:
    print(f'\n=== Domain: {domain.upper()} ===')
    domain_results = {}

    print('  Symmetry...')
    domain_results['symmetry'] = eval_symmetry(domain)
    s = domain_results['symmetry']
    print(f'    acc={s["avg_acc"]:.4f}  cons={s["consistency"]:.4f}  gap={s["gap"]:+.4f}')

    print('  Transitivity...')
    domain_results['transitivity'] = eval_transitivity(domain)
    t = domain_results['transitivity']
    print(f'    acc={t["avg_acc"]:.4f}  cons={t["consistency"]:.4f}  gap={t["gap"]:+.4f}')

    print('  Reflexivity...')
    domain_results['reflexivity'] = eval_reflexivity(domain)
    r = domain_results['reflexivity']
    print(f'    acc={r["avg_acc"]:.4f}  cons={r["consistency"]:.4f}  gap={r["gap"]:+.4f}')

    all_results[domain] = domain_results

print('\nAll evaluations complete.')

In [ ]:
# ── PRINT FULL RESULTS TABLE ──────────────────────────────────────────────────
props = ['reflexivity', 'symmetry', 'transitivity']
domains = list(DOMAINS.keys())

print('=' * 100)
print('ACCURACY vs. CONSISTENCY BY DOMAIN AND PROPERTY — Phi-3-mini-4k-instruct')
print('=' * 100)
print(f'{"Domain":<18}', end='')
for p in props:
    print(f'  {p.capitalize()[:12]:>12} Acc  Cons   Gap', end='')
print()
print('-' * 100)

for d in domains:
    print(f'{d:<18}', end='')
    for p in props:
        r = all_results[d][p]
        print(f'  {r["avg_acc"]:>12.4f} {r["consistency"]:>5.4f} {r["gap"]:>+6.4f}', end='')
    print()

print('=' * 100)
print()
print('Gap = Avg Accuracy - Consistency Score')
print('Positive gap = model is per-question accurate but logically inconsistent')
print('Negative gap = model is consistent but systematically wrong')

In [ ]:
# ── FIGURE 1: Accuracy heatmap (6 domains × 3 properties) ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Phi-3-mini-4k-instruct: Per-Question Accuracy across Domains and Properties',
    fontsize=13, fontweight='bold'
)

metric_label = 'avg_acc'
prop_labels  = ['Reflexivity', 'Symmetry', 'Transitivity']
domain_labels = [d.replace('_', '\n') for d in domains]

for ax, prop, plabel in zip(axes, props, prop_labels):
    vals = np.array([[all_results[d][prop][metric_label]] for d in domains])
    im = ax.imshow(vals, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')
    ax.set_yticks(range(len(domains)))
    ax.set_yticklabels(domain_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_title(plabel, fontsize=12, fontweight='bold')
    for i, d in enumerate(domains):
        v = all_results[d][prop][metric_label]
        ax.text(0, i, f'{v:.3f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if v < 0.55 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('expA_accuracy_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expA_accuracy_heatmap.png')

In [ ]:
# ── FIGURE 2: Accuracy-Consistency Gap heatmap ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Accuracy-Consistency Gap (Acc - Cons): Positive = Model is accidentally correct\n'
    'Phi-3-mini-4k-instruct across Domains and Properties',
    fontsize=12, fontweight='bold'
)

# Diverging colormap centered at 0
norm = mcolors.TwoSlopeNorm(vmin=-0.5, vcenter=0, vmax=0.5)

for ax, prop, plabel in zip(axes, props, prop_labels):
    vals = np.array([[all_results[d][prop]['gap']] for d in domains])
    im = ax.imshow(vals, cmap='RdBu_r', norm=norm, aspect='auto')
    ax.set_yticks(range(len(domains)))
    ax.set_yticklabels(domain_labels, fontsize=10)
    ax.set_xticks([])
    ax.set_title(plabel, fontsize=12, fontweight='bold')
    for i, d in enumerate(domains):
        v = all_results[d][prop]['gap']
        ax.text(0, i, f'{v:+.3f}', ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if abs(v) > 0.2 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('expA_gap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expA_gap_heatmap.png')

In [ ]:
# ── FIGURE 3: Grouped bar chart — Accuracy vs Consistency per domain ──────────
# Focus on transitivity (most interesting gap)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Accuracy vs. Consistency across Domains — Phi-3-mini-4k-instruct\n'
    'Blue = per-question accuracy, Orange = joint logical consistency',
    fontsize=12, fontweight='bold'
)

ACC_COLOR  = '#2196F3'
CONS_COLOR = '#FF5722'
x = np.arange(len(domains))
width = 0.35

for ax, prop, plabel in zip(axes, props, prop_labels):
    accs  = [all_results[d][prop]['avg_acc']     for d in domains]
    conss = [all_results[d][prop]['consistency'] for d in domains]

    bars1 = ax.bar(x - width/2, accs,  width, label='Accuracy',    color=ACC_COLOR,  alpha=0.85)
    bars2 = ax.bar(x + width/2, conss, width, label='Consistency', color=CONS_COLOR, alpha=0.85)

    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7, linewidth=1.2, label='Chance')
    ax.set_xticks(x)
    ax.set_xticklabels([d.replace('_', '\n') for d in domains], fontsize=8)
    ax.set_ylim(0, 1.15)
    ax.set_title(plabel, fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', fontsize=7, color=ACC_COLOR)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', fontsize=7, color=CONS_COLOR)

plt.tight_layout()
plt.savefig('expA_grouped_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: expA_grouped_bars.png')

In [ ]:
# ── KEY ANALYSIS: Is the gap consistent across domains? ───────────────────────
print('KEY ANALYSIS: Transitivity accuracy-consistency gap by domain')
print('(This tests whether failure is structural or domain-specific)')
print()
print(f'{"Domain":<20} {"Accuracy":>10} {"Consistency":>13} {"Gap":>8}')
print('-' * 55)
for d in domains:
    r = all_results[d]['transitivity']
    print(f'{d:<20} {r["avg_acc"]:>10.4f} {r["consistency"]:>13.4f} {r["gap"]:>+8.4f}')
print()

gaps = [all_results[d]['transitivity']['gap'] for d in domains]
print(f'Gap mean:   {np.mean(gaps):+.4f}')
print(f'Gap std:    {np.std(gaps):.4f}')
print(f'Gap range:  {min(gaps):+.4f} to {max(gaps):+.4f}')
print()
if np.std(gaps) < 0.05:
    print('FINDING: Gap is stable across domains (std < 0.05).')
    print('This confirms the failure is STRUCTURAL, not domain-specific.')
else:
    print('FINDING: Gap varies across domains.')
    print('Some domains may introduce additional surface-form effects.')

In [ ]:
# ── Symmetry failure direction analysis ──────────────────────────────────────
# Does the model have a directional bias (always prefers AB over BA)?
print('SYMMETRY FAILURE BREAKDOWN BY DOMAIN')
print('fwd_only = said yes to A→B but no to B→A (directional bias)')
print('rev_only = said yes to B→A but no to A→B (reverse directional bias)')
print('both_no  = said no to both (consistent but wrong)')
print()
print(f'{"Domain":<20} {"both_yes":>9} {"fwd_only":>9} {"rev_only":>9} {"both_no":>9}')
print('-' * 60)
for d in domains:
    ft = all_results[d]['symmetry']['fail_types']
    n  = all_results[d]['symmetry']['n']
    both_yes = ft.get('both_yes', 0)
    fwd_only = ft.get('fwd_only', 0)
    rev_only = ft.get('rev_only', 0)
    both_no  = ft.get('both_no',  0)
    print(f'{d:<20} {both_yes/n:>9.3f} {fwd_only/n:>9.3f} {rev_only/n:>9.3f} {both_no/n:>9.3f}')
print()
print('A large fwd_only rate = Reversal Curse (the model learned A=B but not B=A)')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
# Convert numpy types for JSON serialization
def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj

out = {
    'model': 'microsoft/Phi-3-mini-4k-instruct',
    'n_sym_pairs':    N_SYM,
    'n_trans_triples': N_TRANS,
    'n_refl_entities': N_REFL,
    'results': to_native(all_results),
}

with open('experiment_A_results.json', 'w') as f:
    json.dump(out, f, indent=2)

print('Results saved to experiment_A_results.json')